In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
from bertopic import BERTopic
# ============================================================
# 1) 读取数据
# ============================================================
df = pd.read_csv(r".\output\6-8-customer_complaints_all_model_score.csv")

# 只保留需要的字段
df = df.copy()
df["complaint_type"] = df["complaint_type"].fillna("unknown").astype(str)
df["text_clean"] = df["text_clean"].fillna("").astype(str).str.strip()
df = df[df["text_clean"] != ""].reset_index(drop=True)

df = df[["complaint_id", "complaint_type", "text_clean", "complaint_severity", "resolution_status"]].copy()


d:\miniconda_envs\junliangvenv_nlp1\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [ ]:
# ============================================================
# 2) 先做关键短语抽取（修正版）
# ============================================================
embedding_path = r"D:\huggingface_models\sentence-transformers-all-MiniLM-L6-v2"

sentence_model = SentenceTransformer(
    embedding_path,
    local_files_only=True
)

kw_model = KeyBERT(model=sentence_model)

# 1) 业务通用词 + 语法词：这些不应该当作主题词
# ===========================
# 业务词典
# ===========================
EYEWEAR_DOMAIN = {
    "frame", "frames", "lens", "lenses", "bridge", "bridges",
    "temple", "temples", "arm", "arms", "hinge", "hinges",
    "nose", "pad", "pads", "prescription", "rx", "coating",
    "scratch", "scratched", "crack", "cracked", "broken",
    "damage", "damaged", "loose", "tight", "fit", "fitting",
    "blur", "blurry", "fog", "foggy", "warped", "misaligned",
    "size", "wrong", "delayed", "delay", "shipping", "arrived",
    "package", "boxed", "returned", "refund", "charged", "invoice",
    "billing", "duplicate", "credit", "late", "delivery", "driver",
    "address", "lost", "missing", "smart", "smartglasses", "smartglass", "AR", "AI",
    "camera", "display", "audio", "battery", "charging", "connect", "pairing"
}

NEGATIONS = {
    "not", "no", "never", "without", "dont", "don't", "doesnt", "doesn't",
    "didnt", "didn't", "isnt", "isn't", "wasnt", "wasn't", "cannot", "can't",
    "won't", "wont"
}

GENERIC_WORDS = {
    "complaint", "complaints", "customer", "customers", "service", "product", "products",
    "order", "orders", "issue", "issues", "problem", "problems", "item", "items",
    "store", "seller", "company", "experience", "refund", "purchase", "billing",
    "payment", "charge", "charged", "want", "got", "need", "needed", "just",
    "really", "very", "still", "today", "tomorrow", "week", "month", "year",
    "time", "day", "days", "work", "working", "works", "made", "make", "came",
    "come", "said", "tell", "told", "look", "looks", "looked"
}

STOPWORDS = {
    "a", "an", "the", "this", "that", "these", "those", "it", "its", "they", "them",
    "he", "she", "we", "you", "i", "me", "my", "your", "our", "us", "there", "here",
    "is", "are", "was", "were", "be", "been", "being", "am", "do", "does", "did",
    "if", "then", "but", "and", "or", "for", "with", "from", "into", "out", "on",
    "in", "at", "to", "of", "by", "as", "so", "too", "up", "down"
}

def normalize_tokens(tokens):
    out = []
    for t in tokens:
        t = str(t).strip().lower()
        t = re.sub(r"[^a-z0-9']", "", t)
        if not t:
            continue
        if len(t) <= 2:
            continue
        if t in STOPWORDS or t in GENERIC_WORDS:
            continue
        out.append(t)
    return out

def phrase_has_eyegear_signal(tokens):
    tokens = [t for t in tokens if t]
    if not tokens:
        return False

    # 必须至少包含一个行业词
    if any(t in EYEWEAR_DOMAIN for t in tokens):
        return True

    # 或者包含一个带否定/问题语义的关键词
    if any(t in NEGATIONS or t in {"broken", "loose", "tight", "wrong", "scratch", "crack", "foggy", "blur", "late", "missing"} for t in tokens):
        return True

    return False

def clean_phrase_text(x: str) -> str:
    if x is None:
        return ""
    s = str(x).lower()
    s = re.sub(r"[^a-z0-9\\s'\\-]", " ", s)
    s = re.sub(r"\\s+", " ", s).strip()

    tokens = [t for t in re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", s)]
    tokens = normalize_tokens(tokens)

    if not tokens:
        return ""

    # 关键：不要删除否定词，不要 sort
    # 只要包含行业词或问题词，就保留
    if not phrase_has_eyegear_signal(tokens):
        return ""

    # 只保留 2 个词以上
    if len(tokens) < 2:
        return ""

    # 过滤掉只有功能词或黑名单词的短语
    if all(t in STOPWORDS or t in GENERIC_WORDS for t in tokens):
        return ""

    return " ".join(tokens)

def canonicalize_phrase(s: str) -> str:
    # 不再排序，保留原始词序
    toks = [t for t in re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", str(s).lower()) if t]
    return " ".join(toks)

def extract_key_phrases(text: str, top_n: int = 5) -> str:
    text = str(text).strip()
    if not text:
        return ""

    try:
        kws = kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 2),
            stop_words="english",
            use_mmr=False,
            use_maxsum=False,
            top_n=top_n * 3
        )
    except Exception:
        kws = []

    phrases = []
    for kw, _ in kws:
        cleaned = clean_phrase_text(kw)
        if cleaned:
            phrases.append(cleaned)

    # fallback：提取文本中的高信息词
    if not phrases:
        fallback_tokens = [t for t in re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text.lower()) if t]
        fallback_tokens = normalize_tokens(fallback_tokens)
        if phrase_has_eyegear_signal(fallback_tokens):
            phrases.append(" ".join(fallback_tokens[:5]))

    # 去重，但保留原顺序
    seen = set()
    final = []
    for p in phrases:
        norm = canonicalize_phrase(p)
        if not norm or norm in seen:
            continue
        seen.add(norm)
        final.append(norm)

    # 最终强筛：如果不含眼镜/配镜领域词，直接弃用
    final = [
        p for p in final
        if any(tok in EYEWEAR_DOMAIN or tok in NEGATIONS or tok in {"broken", "loose", "tight", "wrong", "scratch", "crack", "foggy", "blur", "late", "missing"} for tok in p.split())
    ]

    if not final:
        return ""

    return " ".join(final[:top_n])

# 对每条文本先抽关键短语
df["phrase_text"] = df["text_clean"].apply(extract_key_phrases)

# 过滤掉空短语
df = df[df["phrase_text"].str.strip() != ""].reset_index(drop=True)

print("已完成关键短语抽取，样例：")
print(df[["complaint_type", "text_clean", "phrase_text"]].head(5).to_string(index=False))

已完成关键短语抽取，样例：
complaint_type                             text_clean                  phrase_text
      delivery this delay messed up my whole schedule                 delay messed
       product              item is completely warped            completely warped
       product             received the wrong pattern wrong pattern received wrong
       product       battery compartment doesn’t open          battery compartment
       billing     this entire billing cycle is wrong                  cycle wrong


In [3]:
df

,complaint_id,complaint_type,text_clean,complaint_severity,resolution_status,phrase_text
0,2,delivery,this delay messed up my whole schedule,High,Resolved,delay messed
1,3,product,item is completely warped,Low,Unresolved,completely warped
2,6,product,received the wrong pattern,Medium,In Progress,wrong pattern received wrong
3,7,product,battery compartment doesn’t open,Low,Resolved,battery compartment
4,13,billing,this entire billing cycle is wrong,Low,Resolved,cycle wrong
...,...,...,...,...,...,...
2797,7984,delivery,my signature was forged on the delivery confir...,Low,Resolved,forged delivery delivery confirmation
2798,7993,delivery,driver keeps skipping my unit,High,Resolved,driver keeps
2799,7995,billing,the total jumped at checkout without explanation.,Low,Unresolved,total jumped checkout without explanation
2800,7999,delivery,what happened to my next-day delivery?,Low,In Progress,what happened next delivery


In [4]:

# ============================================================
# 3) 按 complaint_type 分组，然后分别做 BERTopic
# ============================================================
all_results = []

for ctype, g in df.groupby("complaint_type", sort=True):
    g_valid = g[g["phrase_text"].fillna("").astype(str).str.strip() != ""].copy()

    if len(g_valid) < 5:
        all_results.append({
            "complaint_type": ctype,
            "topic_id": -1,
            "topic_label": "too_few_samples",
            "topic_size": len(g_valid),
            "top_terms": "n/a",
            "sample_texts": " | ".join(g["text_clean"].head(3).astype(str).tolist())
        })
        continue

    texts = g_valid["phrase_text"].astype(str).tolist()

    topic_model = BERTopic(
        embedding_model=sentence_model,
        language="english",
        calculate_probabilities=True,
        verbose=False,
        nr_topics="auto",
        min_topic_size=max(5, int(len(texts) * 0.05))
    )

    topics, probs = topic_model.fit_transform(texts)
    topic_info = topic_model.get_topic_info()

    for topic_id in sorted(topic_info["Topic"].unique()):
        if topic_id == -1:
            continue

        term_list = topic_model.get_topic(topic_id)
        top_terms = [t[0] for t in term_list[:10]] if term_list else []
        top_terms_str = ", ".join(top_terms)

        topic_mask = np.array(topics) == topic_id
        sample_texts = " | ".join(
            g_valid["text_clean"].iloc[np.where(topic_mask)[0]].head(3).astype(str).tolist()
        )

        all_results.append({
            "complaint_type": ctype,
            "topic_id": int(topic_id),
            "topic_label": str(term_list),
            "topic_size": int(topic_mask.sum()),
            "top_terms": top_terms_str,
            "sample_texts": sample_texts
        })

# 4) 汇总结果
result_df = pd.DataFrame(all_results)

# 5) 生成每个 complaint_type 的子主题表
summary = (
    result_df
    .groupby("complaint_type", as_index=False)
    .agg(
        subtopic_count=("topic_id", "nunique"),
        topic_ids=("topic_id", lambda x: sorted(set(map(int, x)) - {-1})),
        topic_summary=("top_terms", lambda x: " ; ".join([str(v) for v in x if str(v) != "n/a"]))
    )
)

print("\n=== 按 complaint_type 的主题汇总 ===")
print(summary.to_string(index=False))

for ctype in sorted(summary["complaint_type"].unique()):
    print(f"\n===== {ctype} =====")
    print(
        result_df[result_df["complaint_type"] == ctype]
        [["topic_id", "topic_size", "top_terms", "sample_texts"]]
        .sort_values(["topic_size", "topic_id"], ascending=[False, True])
        .to_string(index=False)
    )


=== 按 complaint_type 的主题汇总 ===
complaint_type  subtopic_count             topic_ids                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 topic_summary
       billing               7 [0, 1, 2, 3, 4, 5, 6]             plan, used, bill, never, filled, billed, applied, discount, someone, without ; charging, stop, renewal, advance, keeps, old, card, didn, features, left ; wrong, cycle, billed, currency, user, department, person, address, one, recurring ; without, account, notice, warning, nev

In [6]:
# ============================================================
# 主题命名映射：按 complaint_type + topic_id
# ============================================================
BILLING_TOPIC_NAMES = {
    0: "unauthorized / incorrect billing",
    1: "unexpected / continued charges",
    2: "billing cycle / account information error",
    3: "automatic renewal / renewal notice issue",
    4: "billing notification / transaction issue",
    5: "invoice / billing statement error",
    6: "incorrect plan / package charge"
}

DELIVERY_TOPIC_NAMES = {
    0: "driver behavior / delivery attempt issue",
    1: "delivery delay / in-transit issue",
    2: "delivered but not received",
    3: "delivery status / tracking issue",
    4: "incorrect delivery location / proof-of-delivery issue",
    5: "package damage / improper placement"
}

PRODUCT_TOPIC_NAMES = {
    0: "product condition / possible return issue",
    1: "product damage / physical defect",
    2: "wrong size / color / specification",
    3: "product contamination / severe arrival damage",
    4: "missing charger / cable / accessories",
    5: "instruction / language / labeling issue",
    6: "missing parts / components"
}

TOPIC_NAME_MAP = {
    "billing": BILLING_TOPIC_NAMES,
    "delivery": DELIVERY_TOPIC_NAMES,
    "product": PRODUCT_TOPIC_NAMES,
}

def map_topic_name(ctype: str, topic_id: int) -> str:
    ctype_key = str(ctype).lower()
    topic_id = int(topic_id)
    if ctype_key in TOPIC_NAME_MAP and topic_id in TOPIC_NAME_MAP[ctype_key]:
        return TOPIC_NAME_MAP[ctype_key][topic_id]
    return f"topic_{topic_id}"

In [7]:
# ============================================================
# 3.2) 按 complaint_type 分组，然后分别做 BERTopic
# ============================================================
all_results = []
cluster_assignments = []

for ctype, g in df.groupby("complaint_type", sort=True):
    g_valid = g[g["phrase_text"].fillna("").astype(str).str.strip() != ""].copy()

    if len(g_valid) < 5:
        for idx in g_valid.index:
            cluster_assignments.append({
                "row_id": idx,
                "complaint_type": ctype,
                "bertopic_topic_id": -1,
                "bertopic_topic_name": "too_few_samples"
            })

        all_results.append({
            "complaint_type": ctype,
            "topic_id": -1,
            "topic_name": "too_few_samples",
            "topic_label": "too_few_samples",
            "topic_size": len(g_valid),
            "top_terms": "n/a",
            "sample_texts": " | ".join(g["text_clean"].head(3).astype(str).tolist())
        })
        continue

    texts = g_valid["phrase_text"].astype(str).tolist()

    topic_model = BERTopic(
        embedding_model=sentence_model,
        language="english",
        calculate_probabilities=True,
        verbose=False,
        nr_topics="auto",
        min_topic_size=max(5, int(len(texts) * 0.05))
    )

    topics, probs = topic_model.fit_transform(texts)
    topic_info = topic_model.get_topic_info()

    # 先建立 topic_id -> topic_name 的映射（BERTopic 原始主题词）
    topic_name_map = {}
    for topic_id in sorted(topic_info["Topic"].unique()):
        if topic_id == -1:
            continue
        term_list = topic_model.get_topic(topic_id)
        if term_list:
            top_terms = [t[0] for t in term_list[:5]]
            topic_name_map[int(topic_id)] = ", ".join(top_terms)
        else:
            topic_name_map[int(topic_id)] = f"topic_{topic_id}"

    # 回写每条样本到 cluster_assignments
    for local_idx, topic_id in enumerate(topics):
        row_id = g_valid.index[local_idx]
        real_topic_name = map_topic_name(ctype, int(topic_id))
        cluster_assignments.append({
            "row_id": row_id,
            "complaint_type": ctype,
            "bertopic_topic_id": int(topic_id),
            "bertopic_topic_name": real_topic_name
        })

    # 汇总到 all_results
    for topic_id in sorted(topic_info["Topic"].unique()):
        if topic_id == -1:
            continue

        term_list = topic_model.get_topic(topic_id)
        top_terms = [t[0] for t in term_list[:10]] if term_list else []
        top_terms_str = ", ".join(top_terms)

        topic_mask = np.array(topics) == topic_id
        sample_texts = " | ".join(
            g_valid["text_clean"].iloc[np.where(topic_mask)[0]].head(3).astype(str).tolist()
        )

        all_results.append({
            "complaint_type": ctype,
            "topic_id": int(topic_id),
            "topic_name": map_topic_name(ctype, int(topic_id)),
            "topic_label": str(term_list),
            "topic_size": int(topic_mask.sum()),
            "top_terms": top_terms_str,
            "sample_texts": sample_texts
        })


# 4) 汇总结果
result_df = pd.DataFrame(all_results)

# 5) 把聚类结果回写到原始 df
cluster_df = (
    pd.DataFrame(cluster_assignments)
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[["bertopic_topic_id", "bertopic_topic_name"]]
)
df = df.join(cluster_df, how="left")


# 6) 生成每个 complaint_type 的子主题表
summary = (
    result_df
    .groupby("complaint_type", as_index=False)
    .agg(
        subtopic_count=("topic_id", "nunique"),
        topic_ids=("topic_id", lambda x: sorted(set(map(int, x)) - {-1})),
        topic_names=("topic_name", lambda x: " ; ".join(sorted(set(map(str, x))))),
        topic_summary=("top_terms", lambda x: " ; ".join([str(v) for v in x if str(v) != "n/a"]))
    )
)

print("\n=== 按 complaint_type 的主题汇总 ===")
print(summary.to_string(index=False))


print("\n=== 每个 complaint_type 的 topic 明细 ===")
for ctype in sorted(summary["complaint_type"].unique()):
    print(f"\n===== {ctype} =====")
    print(
        result_df[result_df["complaint_type"] == ctype]
        [["topic_id", "topic_name", "topic_size", "top_terms", "sample_texts"]]
        .sort_values(["topic_size", "topic_id"], ascending=[False, True])
        .to_string(index=False)
    )

print("\n=== 追加后的 df 样例 ===")
print(df[["complaint_type", "text_clean", "phrase_text", "bertopic_topic_id", "bertopic_topic_name"]].head(10).to_string(index=False))


=== 按 complaint_type 的主题汇总 ===
complaint_type  subtopic_count             topic_ids                                                                                                                                                                                                                                                                      topic_names                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             topic_summary
       billing               7 [0, 1, 2, 3, 4, 5, 6]        automatic re

In [9]:
# ============================================================
# 6) 保存 parquet
# ============================================================
output_dir = Path(r".\output\bertopic_phrase_by_complaint_type")
output_dir.mkdir(parents=True, exist_ok=True)

def safe_name(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"[^a-z0-9\u4e00-\u9fff]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x or "unknown"

# 1) 保存带聚类标签的原始 df
df_out = df.copy()
for col in ["complaint_type", "phrase_text", "bertopic_topic_name"]:
    if col in df_out.columns:
        df_out[col] = df_out[col].astype(str)

df_path = output_dir / "6-8-1-customer_complaints_with_bertopic.parquet"
df_out.to_parquet(df_path, index=False)
print(f"saved: {df_path}")

# 2) 保存 result_df（每个 topic 的明细）
result_df_out = result_df.copy()
for col in ["complaint_type", "topic_label", "top_terms", "sample_texts"]:
    if col in result_df_out.columns:
        result_df_out[col] = result_df_out[col].astype(str)

result_path = output_dir / "6-8-1-bertopic_phrase_result_df.parquet"
result_df_out.to_parquet(result_path, index=False)
print(f"saved: {result_path}")

# 3) 保存 summary
summary_out = summary.copy()
for col in ["complaint_type", "topic_ids", "topic_summary"]:
    if col in summary_out.columns:
        summary_out[col] = summary_out[col].astype(str)

summary_path = output_dir / "6-8-1-bertopic_phrase_summary.parquet"
summary_out.to_parquet(summary_path, index=False)
print(f"saved: {summary_path}")

# 4) 按 complaint_type 分拆保存
for ctype in sorted(result_df_out["complaint_type"].unique()):
    per_type_df = result_df_out[result_df_out["complaint_type"] == ctype].copy()
    per_type_df = per_type_df.sort_values(["topic_size", "topic_id"], ascending=[False, True]).reset_index(drop=True)

    out_path = output_dir / f"6-8-1-{safe_name(ctype)}.parquet"
    per_type_df.to_parquet(out_path, index=False)
    print(f"saved: {out_path}")

print("\n=== 保存完成 ===")
print("输出目录:", output_dir)

saved: output\bertopic_phrase_by_complaint_type\6-8-1-customer_complaints_with_bertopic.parquet
saved: output\bertopic_phrase_by_complaint_type\6-8-1-bertopic_phrase_result_df.parquet
saved: output\bertopic_phrase_by_complaint_type\6-8-1-bertopic_phrase_summary.parquet
saved: output\bertopic_phrase_by_complaint_type\6-8-1-billing.parquet
saved: output\bertopic_phrase_by_complaint_type\6-8-1-delivery.parquet
saved: output\bertopic_phrase_by_complaint_type\6-8-1-product.parquet

=== 保存完成 ===
输出目录: output\bertopic_phrase_by_complaint_type
